# SHAP + FFA + Reverse Feature Importance → S3 (multiple models)

**Run this notebook after** model training and performance metrics (e.g. after the main calculator workflow notebook’s training and “Compare ALL Models” steps).

## What it does

1. **Runs SHAP + FFA** per **model** (cohort × variant) using `run_shap_ffa_workflow.py`. All model variants are supported: **top**, **base**, **enhanced**, **wisotzkey**, **FULL** (e.g. CHD_top, Combined_wisotzkey):
   - SHAP values on test set
   - Rule extraction from XGBoost JSON
   - Causal responsibility & dashboard data
   - **Reverse Feature Importance**: missed-prediction drivers (support + IR) and feature profile
2. **Saves outputs locally** under `outputs/shap_ffa/{cohort}_{variant}/` (one directory per model). **Default: all variants** (base, enhanced, top, wisotzkey, FULL) per cohort; set `VARIANTS` to a subset if needed.
3. **Uploads to S3** so Lambda (and dashboards) **only retrieve** precomputed results—no heavy computation in Lambda.

**Lambda** returns Reverse FI **per model** in the metadata API; the **Reverse Feature Importance dashboard tab** can show results for a single model or **summarize across all models**.

**Purpose:** Reverse FI **identifies areas where better data is needed to improve the model**—e.g. features that frequently drive over- or under-prediction and where collecting or improving data can reduce errors.

## S3 layout (matches Lambda)

- `{S3_PREFIX}/dashboard_data/{cohort}_{variant}/dashboard_data.json`
- `{S3_PREFIX}/dashboard_data/{cohort}_{variant}/missed_predictions_drivers.json`
- `{S3_PREFIX}/dashboard_data/{cohort}_{variant}/missed_predictions_feature_profile.csv`
- Other FFA/causal outputs under the same prefix as needed.

**Requirements:** Trained models must exist (run training notebook first). AWS credentials for S3 upload.

**Build/deploy:** Use the same artifact list in `risk_dashboard/prepare_lambda_dir_phts.py` so the Lambda directory and S3 stay in sync (dashboard_data, top_causal_factors, Reverse FI: missed_predictions_drivers.json, missed_predictions_feature_profile.csv/.parquet).

**Local/VS Code:** For a non-notebook run, use `python run_shap_ffa_reverse_fi_s3.py` (same workflow; supports `--no-upload`, `--cohort`, `--variant`, `--top-k`).

## 1. Configuration

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# Calculator directory (notebook lives in cohort_analysis/calculator)
CALCULATOR_DIR = Path.cwd()
for candidate in [
    CALCULATOR_DIR,
    CALCULATOR_DIR / "calculator",
    CALCULATOR_DIR / "cohort_analysis" / "calculator",
    CALCULATOR_DIR / "graft-loss" / "cohort_analysis" / "calculator",
]:
    if (candidate / "run_shap_ffa_workflow.py").exists():
        CALCULATOR_DIR = candidate
        break
assert (CALCULATOR_DIR / "run_shap_ffa_workflow.py").exists(), "run_shap_ffa_workflow.py not found (set cwd to calculator or project root)"

# S3: same bucket/prefix as Lambda (Lambda only retrieves)
S3_BUCKET = os.environ.get("PHTS_BUCKET", "jerome-dixon.io")
S3_PREFIX = os.environ.get("S3_PREFIX", "uva/phts-risk-calculator")

COHORTS = ["CHD", "Myocardio", "Combined"]
VARIANTS = ["base", "enhanced", "top", "wisotzkey", "FULL"]  # Default: all variants per cohort. Set to e.g. ["top"] to run only top.
TOP_K = 15
UPLOAD_TO_S3 = True  # Set False to only run workflow locally

print(f"Calculator dir: {CALCULATOR_DIR}")
print(f"S3 bucket: {S3_BUCKET}")
print(f"S3 prefix: {S3_PREFIX}")
print(f"Cohorts: {COHORTS}")
print(f"Variants: {VARIANTS}")
print(f"Models: {[f'{c}_{v}' for c in COHORTS for v in VARIANTS]}")
print(f"Upload to S3: {UPLOAD_TO_S3}")

## 2. Run SHAP + FFA (including Reverse Feature Importance) per model (cohort × variant)

In [ ]:
python_exe = sys.executable
workflow_script = CALCULATOR_DIR / "run_shap_ffa_workflow.py"

for cohort in COHORTS:
    for variant in VARIANTS:
        model_id = f"{cohort}_{variant}"
        print(f"\n{'='*60}")
        print(f"Running SHAP + FFA + Reverse FI for {cohort} ({model_id})")
        print(f"{'='*60}")
        result = subprocess.run(
            [python_exe, str(workflow_script), "--cohort", cohort, "--model-variant", variant, "--top-k", str(TOP_K)],
            cwd=str(CALCULATOR_DIR),
            capture_output=False,
        )
        if result.returncode != 0:
            print(f"Warning: {workflow_script} for {model_id} exited with code {result.returncode}")
        else:
            print(f"Done: {model_id}")

print("\nAll models completed. Outputs are in outputs/shap_ffa/{cohort}_{variant}/")

## 3. Upload outputs to S3 (for Lambda to retrieve)

In [ ]:
if not UPLOAD_TO_S3:
    print("Upload disabled. Set UPLOAD_TO_S3 = True to push to S3.")
else:
    try:
        import boto3
        s3 = boto3.client("s3")
    except Exception as e:
        print(f"Could not create S3 client: {e}")
        s3 = None

    if s3 is None:
        print("S3 upload skipped (boto3 unavailable or credentials missing).")
    else:
        base = CALCULATOR_DIR / "outputs" / "shap_ffa"
        # Same artifacts as prepare_lambda_dir_phts.py (keep in sync for build/deploy)
        files_to_upload = [
            "dashboard_data.json",
            "missed_predictions_drivers.json",
            "missed_predictions_feature_profile.csv",
            "missed_predictions_feature_profile.parquet",
            "ffa_causal_factors.csv",
            "top_causal_factors.csv",
            "combined_shap_importance.csv",
        ]
        uploaded = 0
        for cohort in COHORTS:
            for variant in VARIANTS:
                model_id = f"{cohort}_{variant}"
                local_dir = base / model_id
                s3_prefix_key = f"{S3_PREFIX}/dashboard_data/{model_id}"
                for fname in files_to_upload:
                    path = local_dir / fname
                    if path.exists():
                        key = f"{s3_prefix_key}/{fname}"
                        try:
                            s3.upload_file(str(path), S3_BUCKET, key)
                            print(f"  Uploaded: s3://{S3_BUCKET}/{key}")
                            uploaded += 1
                        except Exception as e:
                            print(f"  Failed {key}: {e}")
        print(f"\nUploaded {uploaded} file(s) to s3://{S3_BUCKET}/{S3_PREFIX}/dashboard_data/")

## 4. Summary

- **Local:** `outputs/shap_ffa/{cohort}_{variant}/` (one dir per model) contains dashboard_data, causal factors, **Reverse FI** (missed_predictions_drivers.json, missed_predictions_feature_profile.csv).
- **S3:** Same artifacts are under `s3://{bucket}/{prefix}/dashboard_data/{cohort}_{variant}/`. Lambda returns Reverse FI **per model** in the metadata API; the dashboard tab can summarize across models.